In [1]:
import gradio as gr
import sqlite3
import datetime
import re
import requests
import json

In [2]:
# ========================================
# STEP 1: DATABASE SETUP
# ========================================
print("🔧 STEP 1: Setting up database...")

def setup_database():
    """Create our booking database with sample data"""
    print("  📄 Creating database file...")
    
    conn = sqlite3.connect('booking_system.db')
    cursor = conn.cursor()
    
    # Remove old table if exists
    cursor.execute('DROP TABLE IF EXISTS bookings')
    
    # Create new table
    cursor.execute('''
    CREATE TABLE bookings (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        date TEXT NOT NULL,
        time TEXT NOT NULL,
        service TEXT NOT NULL,
        status TEXT DEFAULT 'confirmed',
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # Add sample bookings
    sample_data = [
        ('John Doe', '2025-05-25', '10:00', 'Haircut', 'confirmed'),
        ('Jane Smith', '2025-05-26', '14:30', 'Massage', 'confirmed'),
        ('Bob Johnson', '2025-05-27', '11:15', 'Consultation', 'confirmed'),
        ('Alice Brown', '2025-05-28', '09:00', 'Facial', 'confirmed'),
        ('John Wilson', '2025-05-29', '16:00', 'Haircut', 'confirmed')
    ]
    
    cursor.executemany(
        'INSERT INTO bookings (name, date, time, service, status) VALUES (?, ?, ?, ?, ?)',
        sample_data
    )
    
    conn.commit()
    conn.close()
    print("  ✅ Database created with sample bookings!")


🔧 STEP 1: Setting up database...


In [3]:
# ========================================
# STEP 2: TOOLS FUNCTIONS
# ========================================
print("🛠️ STEP 2: Creating tool functions...")

class BookingTools:
    """Our collection of booking tools"""
    
    def __init__(self):
        print("  🔧 Initializing booking tools...")
        self.db_file = 'booking_system.db'
    
    def find_booking_by_name(self, name):
        """TOOL 1: Find bookings by person's name"""
        print(f"  🔍 Searching for bookings with name containing: '{name}'")
        
        try:
            conn = sqlite3.connect(self.db_file)
            cursor = conn.cursor()
            
            # Search for names that contain the search term (partial matching)
            cursor.execute('''
                SELECT name, date, time, service, status 
                FROM bookings 
                WHERE LOWER(name) LIKE LOWER(?) 
                ORDER BY date, time
            ''', (f'%{name}%',))
            
            results = cursor.fetchall()
            conn.close()
            
            print(f"    📊 Found {len(results)} matching bookings")
            return results
            
        except Exception as e:
            print(f"    ❌ Database error: {e}")
            return []
    
    def create_new_booking(self, name, date, time, service):
        """TOOL 2: Create a new booking"""
        print(f"  📝 Creating booking: {name} on {date} at {time} for {service}")
        
        try:
            conn = sqlite3.connect(self.db_file)
            cursor = conn.cursor()
            
            # Check if slot is already taken
            cursor.execute(
                'SELECT id FROM bookings WHERE date = ? AND time = ?',
                (date, time)
            )
            
            if cursor.fetchone():
                conn.close()
                print("    ❌ Time slot already taken")
                return False, "That time slot is already booked"
            
            # Create the booking
            cursor.execute('''
                INSERT INTO bookings (name, date, time, service, status)
                VALUES (?, ?, ?, ?, 'confirmed')
            ''', (name, date, time, service))
            
            conn.commit()
            conn.close()
            
            print("    ✅ Booking created successfully")
            return True, f"Booking confirmed for {name}"
            
        except Exception as e:
            print(f"    ❌ Error creating booking: {e}")
            return False, f"Error creating booking: {e}"
    
    def get_all_bookings(self):
        """TOOL 3: Get all current bookings"""
        print("  📋 Getting all bookings...")
        
        try:
            conn = sqlite3.connect(self.db_file)
            cursor = conn.cursor()
            
            cursor.execute('''
                SELECT name, date, time, service, status 
                FROM bookings 
                ORDER BY date, time
            ''')
            
            results = cursor.fetchall()
            conn.close()
            
            print(f"    📊 Found {len(results)} total bookings")
            return results
            
        except Exception as e:
            print(f"    ❌ Database error: {e}")
            return []

🛠️ STEP 2: Creating tool functions...


In [4]:
# ========================================
# STEP 3: OLLAMA INTEGRATION
# ========================================
print("🤖 STEP 3: Setting up Ollama integration...")

def call_ollama(messages, model="llama3.2"):
    """Call Ollama with proper message structure"""
    print(f"  🧠 Calling Ollama model: {model}")
    print(f"  💬 System message: {messages[0]['content'][:100]}...")
    print(f"  👤 User message: {messages[1]['content']}")
    
    try:
        # Ollama API endpoint
        url = "http://localhost:11434/api/chat"
        
        payload = {
            "model": model,
            "messages": messages,
            "stream": False
        }
        
        print("  📡 Sending request to Ollama...")
        response = requests.post(url, json=payload)
        
        if response.status_code == 200:
            result = response.json()
            ai_response = result['message']['content']
            print(f"  ✅ Ollama response received: {ai_response[:100]}...")
            return ai_response
        else:
            error_msg = f"Ollama API error: {response.status_code}"
            print(f"  ❌ {error_msg}")
            return error_msg
            
    except Exception as e:
        error_msg = f"Failed to connect to Ollama: {str(e)}"
        print(f"  ❌ {error_msg}")
        return error_msg


🤖 STEP 3: Setting up Ollama integration...


In [5]:
# ========================================
# STEP 4: PATTERN MATCHING (Simple AI)
# ========================================
print("🔍 STEP 4: Setting up pattern matching...")

class SimplePatternMatcher:
    """Simple pattern matching before using Ollama"""
    
    def __init__(self, tools):
        self.tools = tools
        print("  🔍 Pattern matcher initialized")
    
    def try_simple_patterns(self, user_message):
        """Try to handle message with simple patterns first"""
        message = user_message.lower().strip()
        print(f"  🔍 Checking simple patterns for: '{message}'")
        
        # PATTERN 1: Find booking by name
        if self._is_search_request(message):
            name = self._extract_name_from_search(message)
            if name:
                print(f"    ✅ Pattern matched: Search for '{name}'")
                return self._handle_search(name)
        
        # PATTERN 2: Show all bookings
        if self._is_show_all_request(message):
            print("    ✅ Pattern matched: Show all bookings")
            return self._handle_show_all()
        
        # PATTERN 3: Create booking (if all info is present)
        booking_info = self._extract_booking_info(message)
        if booking_info:
            print(f"    ✅ Pattern matched: Create booking")
            return self._handle_create_booking(booking_info)
        
        print("    ❌ No simple pattern matched - will use Ollama")
        return None  # No pattern matched, use Ollama
    
    def _is_search_request(self, message):
        """Check if this is a search request"""
        search_words = ['find', 'search', 'look for', 'show me']
        booking_words = ['booking', 'appointment', 'reservation']
        
        has_search = any(word in message for word in search_words)
        has_booking = any(word in message for word in booking_words)
        
        return has_search and has_booking
    
    def _is_show_all_request(self, message):
        """Check if user wants to see all bookings"""
        patterns = ['show all', 'list all', 'all bookings', 'view all', 'see all']
        return any(pattern in message for pattern in patterns)
    
    def _extract_name_from_search(self, message):
        """Extract name from search message"""
        # Try different patterns
        patterns = [
            r'for\s+([a-zA-Z\s]+)',           # "find booking for John Doe"
            r'find\s+([a-zA-Z\s]+?)\s+booking',  # "find John Doe booking"
            r'search\s+([a-zA-Z\s]+)',        # "search John Doe"
            r'show\s+me\s+([a-zA-Z\s]+?)\s+booking'  # "show me John booking"
        ]
        
        for pattern in patterns:
            match = re.search(pattern, message, re.IGNORECASE)
            if match:
                name = match.group(1).strip()
                # Clean up the name
                name = re.sub(r'\b(booking|bookings|appointment|appointments)\b', '', name, flags=re.IGNORECASE).strip()
                if len(name) > 1:
                    return name
        
        return None
    
    def _extract_booking_info(self, message):
        """Try to extract complete booking information"""
        # This is simplified - in real app you'd use more sophisticated parsing
        # For now, just return None to let Ollama handle booking creation
        return None
    
    def _handle_search(self, name):
        """Handle search request"""
        results = self.tools.find_booking_by_name(name)
        return self._format_search_results(results, name)
    
    def _handle_show_all(self):
        """Handle show all request"""
        results = self.tools.get_all_bookings()
        return self._format_all_bookings(results)
    
    def _handle_create_booking(self, booking_info):
        """Handle booking creation"""
        # Implementation for booking creation
        pass
    
    def _format_search_results(self, results, name):
        """Format search results nicely"""
        if not results:
            return f"❌ No bookings found for '{name}'"
        
        response = f"✅ Found {len(results)} booking(s) for '{name}':\n\n"
        for booking in results:
            response += f"👤 Name: {booking[0]}\n"
            response += f"📅 Date: {booking[1]}\n"
            response += f"⏰ Time: {booking[2]}\n"
            response += f"🛠️ Service: {booking[3]}\n"
            response += f"📊 Status: {booking[4]}\n\n"
        
        return response
    
    def _format_all_bookings(self, results):
        """Format all bookings nicely"""
        if not results:
            return "❌ No bookings found in the system"
        
        response = f"✅ All bookings ({len(results)} total):\n\n"
        for i, booking in enumerate(results, 1):
            response += f"{i}. {booking[0]} | {booking[1]} | {booking[2]} | {booking[3]} | {booking[4]}\n"
        
        return response

🔍 STEP 4: Setting up pattern matching...


In [6]:
# ========================================
# STEP 5: MAIN PROCESSING LOGIC
# ========================================
print("🎯 STEP 5: Setting up main processing logic...")

class MainProcessor:
    """Main class that coordinates everything"""
    
    def __init__(self):
        self.tools = BookingTools()
        self.pattern_matcher = SimplePatternMatcher(self.tools)
        print("  🎯 Main processor initialized")
    
    def process_user_message(self, user_message):
        """
        MAIN PROCESSING FLOW:
        1. Try simple pattern matching first
        2. If no pattern matches, use Ollama
        3. If Ollama suggests using a tool, execute it
        4. Return formatted response
        """
        print(f"\n{'='*60}")
        print(f"🎯 PROCESSING: '{user_message}'")
        print(f"{'='*60}")
        
        # STEP 1: Try simple patterns first
        print("📍 STEP 1: Trying simple pattern matching...")
        simple_result = self.pattern_matcher.try_simple_patterns(user_message)
        
        if simple_result:
            print("✅ Simple pattern handled the request!")
            return simple_result
        
        # STEP 2: Use Ollama for complex requests
        print("📍 STEP 2: Using Ollama for complex processing...")
        return self._use_ollama_fallback(user_message)
    
    def _use_ollama_fallback(self, user_message):
        """Use Ollama when simple patterns don't work"""
        
        # Create the system message
        system_message = """You are a helpful booking assistant. You can help users with:

1. FINDING BOOKINGS: When users ask to find/search bookings by name
2. CREATING BOOKINGS: When users want to make new appointments  
3. VIEWING ALL BOOKINGS: When users want to see all appointments
4. GENERAL QUESTIONS: Answer other booking-related questions

For finding bookings, if the user mentions a name, respond with:
TOOL: find_booking
NAME: [extracted name]

For creating bookings, if user provides name, date, time, and service, respond with:
TOOL: create_booking  
NAME: [name]
DATE: [YYYY-MM-DD format]
TIME: [HH:MM format]
SERVICE: [service type]

For viewing all bookings, respond with:
TOOL: show_all

Otherwise, provide a helpful conversational response about booking management."""

        # Create the messages structure
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ]
        
        # Call Ollama
        ollama_response = call_ollama(messages)
        
        # Process Ollama's response
        return self._process_ollama_response(ollama_response, user_message)
    
    def _process_ollama_response(self, ollama_response, original_message):
        """Process what Ollama told us to do"""
        print("  🤖 Processing Ollama's response...")
        print(f"  📝 Ollama said: {ollama_response}")
        
        # Check if Ollama wants us to use a tool
        if "TOOL: find_booking" in ollama_response:
            name = self._extract_name_from_ollama_response(ollama_response)
            if name:
                print(f"  🔧 Ollama wants to search for: '{name}'")
                results = self.tools.find_booking_by_name(name)
                return self.pattern_matcher._format_search_results(results, name)
        
        elif "TOOL: show_all" in ollama_response:
            print("  🔧 Ollama wants to show all bookings")
            results = self.tools.get_all_bookings()
            return self.pattern_matcher._format_all_bookings(results)
        
        elif "TOOL: create_booking" in ollama_response:
            booking_data = self._extract_booking_from_ollama_response(ollama_response)
            if booking_data:
                print(f"  🔧 Ollama wants to create booking: {booking_data}")
                success, message = self.tools.create_new_booking(**booking_data)
                return f"✅ {message}" if success else f"❌ {message}"
        
        # If no tool needed, return Ollama's response
        print("  💬 No tool needed, returning Ollama's conversational response")
        return ollama_response
    
    def _extract_name_from_ollama_response(self, response):
        """Extract name from Ollama's tool response"""
        match = re.search(r'NAME:\s*(.+)', response, re.IGNORECASE)
        return match.group(1).strip() if match else None
    
    def _extract_booking_from_ollama_response(self, response):
        """Extract booking data from Ollama's tool response"""
        data = {}
        
        patterns = {
            'name': r'NAME:\s*(.+)',
            'date': r'DATE:\s*(.+)',
            'time': r'TIME:\s*(.+)',
            'service': r'SERVICE:\s*(.+)'
        }
        
        for key, pattern in patterns.items():
            match = re.search(pattern, response, re.IGNORECASE)
            if match:
                data[key] = match.group(1).strip()
        
        # Return data only if we have all required fields
        if len(data) == 4:
            return data
        return None

🎯 STEP 5: Setting up main processing logic...


In [ ]:
# ========================================
# STEP 6: GRADIO INTERFACE
# ========================================
print("🖥️ STEP 6: Setting up user interface...")

# Initialize everything
setup_database()
processor = MainProcessor()

def chat_handler(message, history):
    """Handle chat messages"""
    if not message.strip():
        return history
    
    # Process the message
    response = processor.process_user_message(message)
    
    # Add to chat history
    history.append((message, response))
    return history

# Create the interface
with gr.Blocks(title="Beginner-Friendly Booking Assistant") as app:
    gr.Markdown("# 🤖 Beginner-Friendly Booking Assistant")
    gr.Markdown("**How it works:** Simple patterns → Ollama fallback → Tool execution → Response")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 💬 Chat with Assistant")
            
            chatbot = gr.Chatbot(height=500, show_label=False)
            user_input = gr.Textbox(
                placeholder="Ask me about bookings...", 
                label="Your Message",
                lines=1
            )
            
            gr.Markdown("### 🧪 Try These Examples:")
            with gr.Row():
                btn1 = gr.Button("Find John's bookings", size="sm")
                btn2 = gr.Button("Show all bookings", size="sm")
                btn3 = gr.Button("I want to book an appointment", size="sm")
        
        with gr.Column():
            gr.Markdown("### 🔍 Processing Steps:")
            gr.Markdown("1. **Pattern Check**: Look for simple patterns")
            gr.Markdown("2. **Ollama Fallback**: Use AI for complex requests")  
            gr.Markdown("3. **Tool Selection**: Choose appropriate tool")
            gr.Markdown("4. **Execution**: Run tool and format response")
            
            gr.Markdown("### 🛠️ Available Tools:")
            gr.Markdown("• `find_booking_by_name()` - Search by name")
            gr.Markdown("• `create_new_booking()` - Make appointment")
            gr.Markdown("• `get_all_bookings()` - List all bookings")
            
            gr.Markdown("### 📊 Current Database:")
            gr.Markdown("• John Doe, Jane Smith, Bob Johnson")
            gr.Markdown("• Alice Brown, John Wilson")
            gr.Markdown("• Dates: 2025-05-25 to 2025-05-29")

    # Handle user input
    user_input.submit(
        fn=chat_handler,
        inputs=[user_input, chatbot],
        outputs=[chatbot]
    ).then(
        lambda: "",  # Clear input box
        outputs=[user_input]
    )
    
    # Handle example buttons
    btn1.click(
        fn=lambda h: chat_handler("Find bookings for John", h),
        inputs=[chatbot],
        outputs=[chatbot]
    )
    
    btn2.click(
        fn=lambda h: chat_handler("Show all bookings", h),
        inputs=[chatbot],
        outputs=[chatbot]
    )
    
    btn3.click(
        fn=lambda h: chat_handler("I want to book an appointment for tomorrow", h),
        inputs=[chatbot],
        outputs=[chatbot]
    )

# ========================================
# STEP 7: RUN THE APPLICATION
# ========================================
print("🚀 STEP 7: Starting the application...")

if __name__ == "__main__":
    print("✅ Everything initialized successfully!")
    print("📡 Make sure Ollama is running: ollama serve")
    print("🤖 Make sure you have llama3.2 model: ollama pull llama3.2")
    app.launch(debug=True, share=False)

/var/folders/l0/86zkjfks5gz5k5rw35hv8n180000gn/T/ipykernel_39554/3951949625.py:31: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500, show_label=False)


🖥️ STEP 6: Setting up user interface...
  📄 Creating database file...
  ✅ Database created with sample bookings!
  🔧 Initializing booking tools...
  🔍 Pattern matcher initialized
  🎯 Main processor initialized
🚀 STEP 7: Starting the application...
✅ Everything initialized successfully!
📡 Make sure Ollama is running: ollama serve
🤖 Make sure you have llama3.2 model: ollama pull llama3.2
* Running on local URL:  http://127.0.0.1:7892
* To create a public link, set `share=True` in `launch()`.



🎯 PROCESSING: 'I want to book an appointment for tomorrow'
📍 STEP 1: Trying simple pattern matching...
  🔍 Checking simple patterns for: 'i want to book an appointment for tomorrow'
    ❌ No simple pattern matched - will use Ollama
📍 STEP 2: Using Ollama for complex processing...
  🧠 Calling Ollama model: llama3.2
  💬 System message: You are a helpful booking assistant. You can help users with:

1. FINDING BOOKINGS: When users ask t...
  👤 User message: I want to book an appointment for tomorrow
  📡 Sending request to Ollama...
  ✅ Ollama response received: That's a great start! However, I need a bit more information from you.

Could you please tell me wha...
  🤖 Processing Ollama's response...
  📝 Ollama said: That's a great start! However, I need a bit more information from you.

Could you please tell me what service you'd like to book for tomorrow? For example, "spa treatment" or "haircut"?

Once we have that information, I can help you create a new booking.
  💬 No tool needed, ret